## ToolCallingAgent
ToolCallingAgent 是一种典型的工具调用型 Agent，其底层基于 ReAct（推理与行动）思想实现。

In [22]:
from smolagents import ToolCallingAgent, OpenAIModel
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 关闭了模型内部思考，防止与ReAct框架本身预设的“思考-行动”循环逻辑产生冲突，影响工具调用的正确执行
# 此处使用 glm官方api,未测试第三方api
model = OpenAIModel(
    model_id="glm-4.7-flash",
    extra_body={"thinking": {
                    "type": "disabled"
                    }
                }
)
agent = ToolCallingAgent(tools=[], model=model)

agent.run("计算1+2+3...+100的和")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 计算1+2+3...+100的和                                                                                            │
│                                                                                                                 │
╰─ OpenAIModel - zai-org/GLM-5.2 ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '1+2+3+...+100 的和为                                   │
│ 5050。\n\n计算方法：使用等差数列求和公式 S = n(n+1)/2 = 100 × 101 / 2 = 5050。'}                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 1+2+3+...+100 的和为 5050。

计算方法：使用等差数列求和公式 S = n(n+1)/2 = 100 × 101 / 2 = 5050。

Final answer: 1+2+3+...+100 的和为 5050。

计算方法：使用等差数列求和公式 S = n(n+1)/2 = 100 × 101 / 2 = 5050。

[Step 1: Duration 2.23 seconds| Input tokens: 970 | Output tokens: 60]

'1+2+3+...+100 的和为 5050。\n\n计算方法：使用等差数列求和公式 S = n(n+1)/2 = 100 × 101 / 2 = 5050。'

并非通过调用工具实现，而是单纯依赖其自身训练数据中内置的知识。

这正好揭示了 ToolCallingAgent 与 CodeAgent 的本质区别：ToolCallingAgent 的能力来源要么是模型自身的预训练知识，要么是开发者为它提供的外部工具。

## 多agent协同

课程中使用了一个第三方的search, 懒得再去注册，我直接使用 tavily。

In [ ]:
from smolagents import Tool

import os
import json
import requests

load_dotenv(override=True)

model = OpenAIModel(
    model_id="zai-org/GLM-5.2",#此处使用 硅基流动api,但看上去效果不太好
    extra_body={"thinking": {
                    "type": "disabled"
                    }
                }
)

class TavilyWebSearch(Tool):
    name = "tavily_web_search"
    description = """
    用于根据搜索词调用 Tavily AI Search API 进行联网搜索"""
    inputs = {
        "query": {
            "type": "string",
            "description": "要搜索的查询",
        }
    }
    output_type = "string"

    def forward(self, query: str) -> str:
        API_KEY = os.getenv("TAVILY_API_KEY")
        data = {
            "api_key": API_KEY,
            "query": query,
            "search_depth": "basic",
            "max_results": 10,
            "include_answer": False,
        }
        endpoint = "https://api.tavily.com/search"
        headers = {
            "Content-Type": "application/json"
        }

        response = requests.post(endpoint, headers=headers, data=json.dumps(data))
        response.raise_for_status()
        search_ret = response.json()
        return self.tavily_for_list(search_ret)

    def tavily_for_list(self, search_ret: dict):
        results = search_ret.get("results", [])
        ret = []
        for item in results:
            ret.append({
                "title": item.get("title"),
                "summary": item.get("content"),
                "url": item.get("url"),
            })
        return ret

使用glm(硅基或者官方api),会先报 code格式对。后面更换其他模型再试下效果。
虽然最后都形成了报告，但 step 太多了，十几个步骤。

测试 tavily工具类

In [33]:
tool = TavilyWebSearch()
result = tool.forward("Claude fable 5 发布时间")

print(result)

[{'title': 'Anthropic 发布Claude Fable 5，官方称其能力“超过我们此前 ...', 'summary': 'Jun 10, 2026 — 6月9日，Anthropic正式发布Claude Fable 5——一款被定位为"Mythos级"的安全可用模型。这是Anthropic首次将Mythos级别的模型"降级"后向公众开放，同时保留 ...Read more', 'url': 'https://www.oschina.net/news/455243'}, {'title': 'Anthropic 發布Claude Fable 5：首款公開開放的Mythos 級AI ...', 'summary': 'Anthropic 於2026 年6 月正式發布Claude Fable 5,這是首款向一般市場開放的Mythos-class(Mythos 級)AI 模型。', 'url': 'https://www.facebook.com/jackshenadvisor/videos/anthropic-%E7%99%BC%E5%B8%83-claude-fable-5%E9%A6%96%E6%AC%BE%E5%85%AC%E9%96%8B%E9%96%8B%E6%94%BE%E7%9A%84-mythos-%E7%B4%9A-ai-%E6%A8%A1%E5%9E%8Banthropic-%E6%96%BC-2026-%E5%B9%B4-6-%E6%9C%88%E6%AD%A3%E5%BC%8F%E7%99%BC%E5%B8%83-clau/1579656500257372'}, {'title': '刚刚，Claude Mythos 5发布！5000万行代码1天搞定', 'summary': 'Jun 10, 2026 — 自家有史以来最强悍的大模型旗舰，分两个版本端上桌：Claude Fable 5与Claude Mythos 5。 Fable 5是加了防护网版本的Mythos**，面向所有用户开放。 一旦用户 ...Read more', 'url': 'https://zhuanlan.zhihu.com/p/2047932995126956032'}, {'title': '刚刚，Claude最强模型Fable 5发布：性能爆炸，价格翻倍

构建一个subagent,用于web search

In [34]:
web_search_agent = ToolCallingAgent(
    name = "web_search_agent",
    description = "可以根据用户的问题，进行联网搜索，返回搜索结果",
    tools=[TavilyWebSearch()],
    model=model, 
    max_steps=10
)

用 CodeAgent 做主agent

In [37]:
from smolagents import CodeAgent
from tools import ReadCSVTool, WriteMDTool

agent = CodeAgent(
    tools=[ReadCSVTool(), WriteMDTool()], 
    model=model, 
    stream_outputs=False,
    managed_agents=[web_search_agent],
    additional_authorized_imports=["*"]
)

Caution: you set an authorization for all imports, meaning your agent can decide to import any package it deems 
necessary. This might raise issues if the package is not installed in your environment.

In [38]:
prompt = """
#角色设定：你是一位专业的金融数据分析师。
#任务目标：请结合本地数据与最新网络资讯，对“燕京啤酒”进行全面的行情与基本面分析，并生成一份 Markdown 格式的投资分析报告。
#具体执行步骤：
1. 本地数据走势分析：
  - 读取并分析文件 data/yanjing_beer_daily_k_20250518_20260518.csv。
  - 提取关键指标（如开盘价、收盘价、最高/最低价、成交量等），分析近一年的股价整体走势、波动特征及关键时间节点。
2. 联网搜索与资讯挖掘：
  - 搜索燕京啤酒近期的财经新闻、公司公告及研报。
  - 重点梳理近期的利好因素（如业绩预增、新品发布、机构评级等）与潜在风险（如市场竞争、资金流向、原材料成本等）。
3. 综合分析与总结：
  - 将本地技术面数据与网络基本面消息相结合，进行交叉验证与统一分析。
  - 给出客观的总结性观点。
4. 输出报告：
  - 整合以上所有分析内容整合成一份结构清晰、排版美观的 Markdown 报告。
  - 将报告内容完整写入当前工作目录下的 report.md 文件中。
"""
agent.run(prompt)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ #角色设定：你是一位专业的金融数据分析师。                                                                       │
│ #任务目标：请结合本地数据与最新网络资讯，对“燕京啤酒”进行全面的行情与基本面分析，并生成一份 Markdown            │
│ 格式的投资分析报告。                                                                                            │
│ #具体执行步骤：                                                                                                 │
│ 1. 本地数据走势分析：                                                                                           │
│   - 读取并分析文件 data/yanjing_beer_daily_k_20250518_20260518.csv。                                            │
│   - 提取关键指标（如开盘价、收盘价、最高/最低价、成交量等），分析近一年的股价整体走势、波动特征及关键时间节点。 │
│ 2. 联网搜索与资讯挖掘：                                                                                         │
│   - 搜索燕京啤酒近期的财经新闻、公司公告及研报。                                                                │
│   -                                                                                                             │
│ 重点梳理近期的利好因素（如业绩预增、新品发布、机构评级等）与潜在风险（如市场竞争、资金流向、原材料成本等）。    │
│ 3. 综合分析与总结：                                                                                             │
│   - 将本地技术面数据与网络基本面消息相结合，进行交叉验证与统一分析。                                            │
│   - 给出客观的总结性观点。                                                                                      │
│ 4. 输出报告：                                                                                                   │
│   - 整合以上所有分析内容整合成一份结构清晰、排版美观的 Markdown 报告。                                          │
│   - 将报告内容完整写入当前工作目录下的 report.md 文件中。                                                       │
│                                                                                                                 │
╰─ OpenAIModel - zai-org/GLM-5.2 ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
            Here is your code snippet:
            Thought: I'll start by analyzing the task. I need to:
1. Read and analyze the local CSV file for Yanjing Beer daily K-line data
2. Search online for recent financial news, company announcements, and research reports
3. Combine technical and fundamental analysis
4. Generate a Markdown report and write it to report.md

Let me begin by reading the local CSV file to understand the data structure and analyze the price trends.</code>
            Make sure to include code with the correct pattern, for instance:
            Thoughts: Your thoughts
            <code>
            # Your python code here
            </code>
Make sure to provide correct code blobs.

[Step 1: Duration 3.36 seconds| Input tokens: 2,484 | Output tokens: 89]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 14 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("燕京啤酒（000729.SZ）投资分析报告已成功生成并写入 report.md 文件！\n\n报告完成情况总结：\n\n## 📊  
  任务完成情况\n\n**1. 本地数据走势分析（技术面）**\n- 数据周期：2025-05-19 至 2026-05-18（242个交易日）\n-        
  近一年收益率：-2.37%，最大回撤：-20.54%\n- 年内最高价：14.56元（2026-04-07），最低价：11.16元（2025-12-30）\n-   
  股价经历四个阶段：冲高回落→持续下行→强势反弹→冲高回落\n- 当前技术面：均线空头排列，RSI                           
  37.45接近超卖，短期处于调整阶段\n\n**2. 联网搜索与基本面分析**\n-                                                
  2025年全年：营收153.33亿元（+4.54%），净利润16.79亿元（+59.06%）\n-                                              
  2026年Q1：营收40.97亿元（+7.06%），净利润2.65亿元（+60.19%）\n-                                                  
  核心大单品U8销量90万千升（+29.31%），新品A10全麦拉格上市\n-                                                      
  23家机构一致看好（买入18家，增持5家），目标均价15.21元\n- 毛利率持续提升至46.32%，成本红利持续\n\n**3.           
  综合分析结论**\n- 基本面强劲增长与技术面短期调整形成显著背离\n-                                                  
  当前估值合理偏低（PEG约0.56），机构目标价有23%上行空间\n- 中长期投资价值显著，短期需关注技术面企稳信号\n\n**4.   
  报告输出**\n- 文件路径：report.md\n- 文件大小：17,193 bytes\n-                                                   
  报告结构：8个章节（摘要、公司概况、技术面分析、基本面分析、综合分析、风险提示、投资建议、免责声明）\n\n报告内容  
  完整，已成功写入当前工作目录下的 report.md 文件。")                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 燕京啤酒（000729.SZ）投资分析报告已成功生成并写入 report.md 文件！

报告完成情况总结：

## 📊 任务完成情况

**1. 本地数据走势分析（技术面）**
- 数据周期：2025-05-19 至 2026-05-18（242个交易日）
- 近一年收益率：-2.37%，最大回撤：-20.54%
- 年内最高价：14.56元（2026-04-07），最低价：11.16元（2025-12-30）
- 股价经历四个阶段：冲高回落→持续下行→强势反弹→冲高回落
- 当前技术面：均线空头排列，RSI 37.45接近超卖，短期处于调整阶段

**2. 联网搜索与基本面分析**
- 2025年全年：营收153.33亿元（+4.54%），净利润16.79亿元（+59.06%）
- 2026年Q1：营收40.97亿元（+7.06%），净利润2.65亿元（+60.19%）
- 核心大单品U8销量90万千升（+29.31%），新品A10全麦拉格上市
- 23家机构一致看好（买入18家，增持5家），目标均价15.21元
- 毛利率持续提升至46.32%，成本红利持续

**3. 综合分析结论**
- 基本面强劲增长与技术面短期调整形成显著背离
- 当前估值合理偏低（PEG约0.56），机构目标价有23%上行空间
- 中长期投资价值显著，短期需关注技术面企稳信号

**4. 报告输出**
- 文件路径：report.md
- 文件大小：17,193 bytes
- 报告结构：8个章节（摘要、公司概况、技术面分析、基本面分析、综合分析、风险提示、投资建议、免责声明）

报告内容完整，已成功写入当前工作目录下的 report.md 文件。

[Step 14: Duration 13.38 seconds| Input tokens: 203,812 | Output tokens: 10,937]

'燕京啤酒（000729.SZ）投资分析报告已成功生成并写入 report.md 文件！\n\n报告完成情况总结：\n\n## 📊 任务完成情况\n\n**1. 本地数据走势分析（技术面）**\n- 数据周期：2025-05-19 至 2026-05-18（242个交易日）\n- 近一年收益率：-2.37%，最大回撤：-20.54%\n- 年内最高价：14.56元（2026-04-07），最低价：11.16元（2025-12-30）\n- 股价经历四个阶段：冲高回落→持续下行→强势反弹→冲高回落\n- 当前技术面：均线空头排列，RSI 37.45接近超卖，短期处于调整阶段\n\n**2. 联网搜索与基本面分析**\n- 2025年全年：营收153.33亿元（+4.54%），净利润16.79亿元（+59.06%）\n- 2026年Q1：营收40.97亿元（+7.06%），净利润2.65亿元（+60.19%）\n- 核心大单品U8销量90万千升（+29.31%），新品A10全麦拉格上市\n- 23家机构一致看好（买入18家，增持5家），目标均价15.21元\n- 毛利率持续提升至46.32%，成本红利持续\n\n**3. 综合分析结论**\n- 基本面强劲增长与技术面短期调整形成显著背离\n- 当前估值合理偏低（PEG约0.56），机构目标价有23%上行空间\n- 中长期投资价值显著，短期需关注技术面企稳信号\n\n**4. 报告输出**\n- 文件路径：report.md\n- 文件大小：17,193 bytes\n- 报告结构：8个章节（摘要、公司概况、技术面分析、基本面分析、综合分析、风险提示、投资建议、免责声明）\n\n报告内容完整，已成功写入当前工作目录下的 report.md 文件。'

## 思考
构建多 Agent 系统时，有没有必要让主 Agent 不做任何业务，而是为其添加多个子 CodeAgent、ToolCallingAgent来处理业务，主 Agent 只做调度与结果整理？

### Yes
1. Action Space会直接影响 LLM 的规划质量

subagent本质上被包装称一个‘工具’，主agent只能看到它的 name+description+输入/输出。如果主agent自己还挂了一堆业务工具，它的system prompt中工具列表会变得又长又杂，LLM在‘该分解任务、调用subagent’ 还是 ‘直接调用工具’之间会更容易选错，或者列出不必要的中间步骤。工具越多、越异构，单个agent的判断准确率下降越明显。工具数量保持精简。
> 注：强化学习/决策系统的通用术语，指"一个决策主体在某一时刻可以选择的所有可能动作的集合"。放到 LLM Agent 语境下，action space 就是这个 Agent 在每一步推理时，可以选择调用的工具/函数/子 Agent 的集合。langgraph和 Claude code sdk中没有针对性的名字，类似的 langgraph bind_tools 作为tools集合，CC sdk中 allowed_tools.
2. 关注点分离，便于独立测试和替换。

每个subagent只服务一个业务域，可以单独写 eval、单独调 prompt、甚至单独换模型（比如给做简单信息抽取的子 Agent 配便宜的小模型，给需要复杂推理的子 Agent 配更强的模型）

3. 权限与安全边界更清晰

比如一个子 CodeAgent 专门跑数据库查询、一个专门跑网络请求，权限（可执行的 import、可访问的 API key）可以按子 Agent 隔离，主 Agent 不直接持有这些敏感能力。

### No
1. 对于小系统，是过度设计

如果整个系统工具数量本来就不多（比如 5-8 个），单个 CodeAgent 完全可以胜任，没必要为了"架构上好看"而拆分。拆分带来的调试复杂度（日志分散在多个 agent trace 里）反而增加维护成本。

2. 额外的 LLM 调用开销

每多一层子 Agent，就多一轮完整的 LLM 推理（甚至多轮，因为子 Agent 自己也可能要多步 ReAct 循环）。对于简单、高频、确定性强的业务（比如"查一下天气""算个数"），直接在主 Agent 挂 1-2 个工具处理，比包一层子 Agent 再转发要快得多、便宜得多。

3. 结果传递会丢信息

子 Agent 最终通过 final_answer 返回一段文本/结构化结果给 manager，中间过程的丰富上下文（比如工具调用的原始数据）默认不会带回去。如果主 Agent"纯调度、不碰业务"，就必须完全信任子 Agent 总结的结果，一旦子 Agent 总结不到位，主 Agent 没有能力直接核查或补救，只能再发一轮指令让子 Agent 重跑——这在需要精确数据传递的场景里是个坑。示例见下面cell.


### 混合模式
按"是否需要复杂多步推理"来划分，而不是"是否有必要"来一刀切

1. 主 Agent 直接持有少量高频、简单、结果不需要二次判断的工具（比如查天气、查日历这种一步到位的），减少不必要的分层。
2. 只把"需要多步骤规划、需要独立上下文、需要专业工具集（比如写代码执行、复杂检索）"的业务下沉给子 Agent。
3. 子 Agent 的数量控制在主 Agent 能清晰区分职责的范围内。subagent数量没有量化标准，经验参考 5～10个以内、进行语义区分度测试、跑eval、检查职责是否有重叠。详细见下面cell.

### 为什么subagent完成同样的任务，成本比直接调用tool高

- 直接调用tool： 主agent 1次推理 -> 调用tool -> 主agent消化tool的结果并决定是否final anwser
- 调用subagent：主agent 1次推理 -> 调用subagent,subagent推理并决定调用tool -> 调用tool -> subagent消化tool的结果并决定是否final anwser，假设是 返回主agent -> 主agent消化subagent的结果

可看到调用subagent至少多了2次推理(suabagent收到任务时1次推理，完成任务前消化结果的1次推理)。

假设一次 LLM 调用平均 1 秒、消耗 1K token。则每次subagent会至少多花费2秒时间、2K token.(粗略假设)

### "结果传递会丢信息"的示例

例如，一个"宏观研究助手"系统。
- 主 Agent：负责整理一份给客户的利率展望简报
- 子 Agent（research_agent）：负责搜网页、查财经数据、总结美联储最新决议

如果主 Agent"零业务、完全信任子 Agent 总结"，子 Agent 跑完自己的多步搜索循环后，通过 final_answer 只返回一段总结文本给主 Agent，比如："美联储维持利率不变，市场预期偏鹰派"。

问题就出在这里——子 Agent 内部搜索到的原始数据（具体区间 3.50%-3.75%、点阵图中位数 3.8% vs 3月的 3.4%、会议具体日期 6月17日、数据来源 TradingKey/CME）在它自己总结成一句话的过程中被压缩丢弃了。主 Agent 因为自己"不碰业务"，看不到子 Agent 的中间步骤（除非专门配置了 memory 传递机制），只能拿着这句模糊总结去写简报，最终客户收到的可能是"美联储维持利率不变，态度偏鹰"这种**没有具体数字、没有数据来源、无法核查**的表述——**如果客户追问"具体区间是多少""点阵图预期是几"，主 Agent 答不出来，只能重新触发子 Agent 再跑一遍**。

这在需要精确数值、时间戳、数据来源的业务场景里（金融、医疗、法律合规）是实打实的风险点。解法通常是：让子 Agent 的 final_answer 返回结构化数据（比如 JSON：{rate_range: "3.50-3.75%", dot_plot_median: "3.8%", meeting_date: "2026-06-17", source: "TradingKey"}）而不是自然语言总结，这样主 Agent 拿到的是可核查、可复用的原始信息，而不是被压缩过的一句话。

### subAgent /tool 数量到底怎么判断，有没有量化标准

(1) 语义可区分度测试（最实用的一条）
把所有子 Agent 的 name + description 罗列出来，问自己："如果我是这个主 Agent 的 LLM，只看这些描述，我能不能不假思索地把 10 个不同类型的用户请求正确路由到对应子 Agent？"如果有两个子 Agent 的描述读起来相似、容易混淆（比如"处理数据分析"和"处理数据统计"），那不管数量多少，都已经超出了主 Agent 能可靠区分的范围，需要**合并或重写** description。

(2) 经验参考区间
业界一般共识是：单层挂载的工具/子 Agent 总数控制在 **5-10 个以内**，超过这个数量，路由错误率会明显上升（这和 LLM 的上下文利用、注意力分散有关，工具越多，system prompt 越长，模型越容易"选错"或"漏选"）。如果业务确实需要更多子 Agent，通常做法不是让主 Agent 直接管理十几个子 Agent，而是引入中间层——比如按业务域先分成几个"**域管理 Agent**"（如"数据类管理者""执行类管理者"），主 Agent 只管理这几个域管理者，域管理者再各自管理自己的 3-5 个子 Agent，做成**树状结构**而不是扁平结构。

(3) 实测路由准确率
最靠谱的判断标准其实不是"数一数有几个"，而是**跑 eval**：准备一批有代表性的**真实用户请求**，跑一遍看主 Agent 路由到子 Agent 的准确率。如果准确率明显下降（比如低于 90%），就是数量或描述已经超出主 Agent 的可靠判断范围，需要精简或重新分组，而不是死守某个数字。

(4) 职责是否有重叠
如果两个子 Agent 处理的任务在业务逻辑上有交集（比如一个"财务分析 Agent"和一个"市场分析 Agent"都可能被问到"这季度收入怎么样"），无论描述写得多清楚，主 Agent 都会在部分请求上纠结甚至选错。这种情况下应该先解决"职责边界是否清晰"的问题，再谈数量。

一句话总结：没有绝对数字标准，判断标准是"主 Agent 能否稳定、准确地路由"，5-10 个是经验上限，超过就该考虑分层而不是继续平铺。

### 有待改进之处

代码为学习示例代码，非生产可用代码。有待改进之处：
1. 增加agent trace
2. 上面明显可以看到 自主生成代码 流程的质量严重依赖LLM的推理能力，若使用弱一点模型，推理step明显增加，其实最终算下来的成本未必低，且时间成本明显增加。优先选择代码能力强的模型(或使用专门为code任务微调过的模型)。
3. 增加 eval
4. 监控&告警
5. agent & subagent 的fallback机制